# Gemini Inference Manifest Generation

This Colab notebook is designed to prepare Gemini's transcription manifest for evaluation. It merges raw transcription results with ground-truth segmentations.

**Note:** This version is focused strictly on data merging and manifest generation. Analysis (WER calculation) and visualization are handled in a separate benchmark notebook.

The Colab performs the following functions:

1.  **Loads existing ground truth** from the baseline manifest.
2.  **Maps Gemini segments** using the `batch_manifest.jsonl` to align raw API results with the benchmark offsets.
3.  **Generates a merged manifest** specifically containing the Gemini predictions.
4.  **Exports the final benchmark** back to Google Cloud Storage.

Note: This Colab reads/writes content in the `wd-transcription-data` bucket in GCP.

In [ ]:
# @title Install dependencies
!pip install -q loguru

In [ ]:
import collections
import json
import re
from typing import Any

from google.cloud import storage
from google.colab import auth
from loguru import logger

# @markdown ### GCP Configuration
GCP_PROJECT_ID = "" # @param {type:"string"}
GCS_BUCKET = "" # @param {type:"string"}

# Model Info
MODEL_ID = "gemini-3.1-pro-preview"
MODEL_VERSION = re.sub(r"[-\.]", "_", MODEL_ID)

# Input/Output Paths
GCS_OUTPUT_DIR = "transcripts/one_hour_pilot_audio"
GCS_MAP_DIR = "segmented_audio/one_hour_pilot_audio"
GCS_MANIFEST_PATH = "manifests/one_hour_pilot_transcriptions.json"
GCS_FINAL_OUTPUT_DIR = "inference_manifests"

# Filenames
BATCH_MANIFEST_FILENAME = "batch_manifest.jsonl"
EXISTING_BENCHMARK_FILENAME = "one_hour_pilot_transcriptions.json"
UPDATED_BENCHMARK_FILENAME = f"one_hour_pilot_{MODEL_VERSION}.jsonl"

# Authenticate to GCS
auth.authenticate_user()
!gcloud config set project {GCP_PROJECT_ID} --quiet

In [ ]:
# @title Helper functions
from pathlib import Path


def load_manifest(path: str) -> list[dict[str, Any]]:
    """Helper to load manifest. Cleans up formatting issues and stray linefeeds."""
    data = []
    if not Path(path).exists():
        logger.error(f"Manifest path not found: {path}")
        return []

    with open(path, encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line:
                continue
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return data

def merge_gcs_results_to_manifest(
    baseline_data: list[dict[str, Any]],
    batch_manifest_data: list[dict[str, Any]],
    gcs_bucket_name: str,
    output_file: str,
    predictions_gcs_path: str | None = None
) -> dict[str, Any]:
    client = storage.Client(project=GCP_PROJECT_ID)
    bucket = client.bucket(gcs_bucket_name)

    gemini_predictions = {}

    if predictions_gcs_path:
        logger.info(f"Loading combined predictions from gs://{gcs_bucket_name}/{predictions_gcs_path}")
        local_preds = "temp_predictions.jsonl"
        bucket.blob(predictions_gcs_path).download_to_filename(local_preds)
        with open(local_preds) as f:
            for raw_line in f:
                line = raw_line.strip()
                if not line:
                    continue
                try:
                    data = json.loads(line)
                    req_parts = data.get("request", {}).get("contents", [{}])[0].get("parts", [])
                    file_uri = next((p["file_data"]["file_uri"] for p in req_parts if "file_data" in p), "")

                    match = re.search(r"([^/]+)__seg(\d{3})", file_uri)
                    if not match:
                        continue
                    example_id, seg_key = match.group(1), match.group(2)

                    candidates = data.get("response", {}).get("candidates", [])
                    if not candidates:
                        continue

                    raw_text = candidates[0].get("content", {}).get("parts", [{}])[0].get("text", "")
                    inner_data = json.loads(raw_text)
                    gemini_predictions[(example_id, seg_key)] = inner_data.get("transcription", "").strip()
                except Exception as e:
                    logger.warning(f"Failed to parse prediction line: {e}")
                    continue

    offset_to_seg = collections.defaultdict(dict)
    for entry in batch_manifest_data:
        offset_to_seg[entry.get("example_id", "")][float(entry.get("offset", 0.0))] = entry.get("segment_id", "")

    merged_records = []
    matched_count = 0
    missing_count = 0
    EPSILON = 0.001

    for b_info in baseline_data:
        example_id = Path(b_info["audio_filepath"]).stem
        b_offset = float(b_info.get("offset", 0.0))
        gemini_text = ""
        matched_seg_id = None

        available_offsets = offset_to_seg.get(example_id, {})
        for off_val, s_id in available_offsets.items():
            if abs(off_val - b_offset) <= EPSILON:
                matched_seg_id = s_id
                break

        if matched_seg_id and (example_id, matched_seg_id) in gemini_predictions:
            gemini_text = gemini_predictions[(example_id, matched_seg_id)]
            matched_count += 1
        else:
            missing_count += 1

        merged_records.append({**b_info, f"pred_text_{MODEL_VERSION}": gemini_text})

    with open(output_file, "w", encoding="utf-8") as f_out:
        f_out.writelines(json.dumps(rec) + "\n" for rec in merged_records)

    return {"total": len(baseline_data), "matched": matched_count, "missing": missing_count}

def run_pipeline() -> None:
    client = storage.Client(project=GCP_PROJECT_ID)
    bucket = client.bucket(GCS_BUCKET)
    PREDICTIONS_FILE = "transcripts/one_hour_pilot_audio/gemini_3_1_pro_preview/predictions.jsonl"

    bucket.blob(GCS_MANIFEST_PATH).download_to_filename(EXISTING_BENCHMARK_FILENAME)
    baseline_data = load_manifest(EXISTING_BENCHMARK_FILENAME)

    batch_manifest_path = f"{GCS_MAP_DIR}/{BATCH_MANIFEST_FILENAME}"
    bucket.blob(batch_manifest_path).download_to_filename(BATCH_MANIFEST_FILENAME)
    batch_manifest_data = load_manifest(BATCH_MANIFEST_FILENAME)

    stats = merge_gcs_results_to_manifest(baseline_data, batch_manifest_data, GCS_BUCKET, UPDATED_BENCHMARK_FILENAME, PREDICTIONS_FILE)

    logger.info("--- Processing Summary ---")
    logger.info(f"Total Baseline Rows: {stats['total']}")
    logger.info(f"Successfully Matched: {stats['matched']}")
    logger.info(f"Missing/Unmatched:    {stats['missing']}")

    gcs_output_path = f"{GCS_FINAL_OUTPUT_DIR}/{UPDATED_BENCHMARK_FILENAME}"
    bucket.blob(gcs_output_path).upload_from_filename(UPDATED_BENCHMARK_FILENAME)
    logger.info(f"Uploaded final manifest to gs://{GCS_BUCKET}/{gcs_output_path}")

In [ ]:
# @title Run Processing
run_pipeline()